# Identity consistency — four case-study accounts

Looks for first-person identity statements that contradict each other within one account: age, sex or
gender, occupation, location or nationality, voting and party, religion, ethnicity, family, education,
military service.

1. Load each account's full post and comment history (`p2_data_cleaning/data/case_study_history/`).
2. **Candidate extraction (this notebook).** Regexes catch first-person identity statements. They are
   deliberately broad: recall matters here, and every candidate is read by a person/model afterwards.
3. **Review (outside this notebook).** Candidates are read in context; each real self-claim is recorded
   with its id, and claims are compared within each category. A contradiction is reported only after the
   full text of both items has been re-read and quotes, hypotheticals, jokes and talk about other people
   ruled out.

Outputs in `identity_consistency_output/` (new files only). v1 outputs missed "I'm in <place>" and matched any word after "here in"; v2 fixes both.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import gzip
import json
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Raw full histories of the four case-study accounts, and the output folder.
RAW = os.path.join('..', 'p2_data_cleaning', 'data', 'case_study_history', 'raw')
OUT_DIR = 'identity_consistency_output'
ACCOUNTS = ['CaseA', 'CaseB', 'CaseC', 'CaseD']
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# One row per post or comment of an account; a post's text is its title plus body.
def load(account):
    rows = []
    for kind in ('posts', 'comments'):
        path = os.path.join(RAW, kind, f'{account.lower()}.jsonl.gz')
        for line in gzip.open(path, 'rt', encoding='utf-8'):
            r = json.loads(line)
            text = (f"{r.get('title', '')}\n\n{r.get('selftext', '')}" if kind == 'posts' else r.get('body', '')) or ''
            rows.append(dict(account=account, kind=kind[:-1], id=r['id'], created_utc=int(r['created_utc']),
                             subreddit=r.get('subreddit'), link_id=r.get('link_id'),
                             permalink=r.get('permalink'), text=text))
    return pd.DataFrame(rows)

# All four accounts; drop empty, deleted and removed items. Counts by account and kind.
items = pd.concat([load(a) for a in ACCOUNTS], ignore_index=True)
items = items[~items['text'].str.strip().isin(['', '[deleted]', '[removed]'])]
items.groupby(['account', 'kind']).size().unstack()

## Candidate patterns
Each pattern needs a first-person anchor (I / I'm / my / me / as a …), so talk about third parties is
mostly excluded. Lines starting with `>` are Reddit quotes and are dropped before matching.

In [ ]:
# I = a first-person anchor. Every pattern below needs one, so statements about other people are mostly excluded.
I = r"(?:\bI(?:'m| am|'ve| have| was|'d| had)?\b|\bmy\b|\bme\b|\bas an?\b)"
# Regex patterns for self-descriptions, by category.
PATTERNS = {
    'age': [r"\bI(?:'m| am)\s+(?:a\s+|an\s+)?(?:\d{2})\b(?!\s*(?:%|percent|minutes|hours|days|mph|lbs|kg|k\b|dollars|times|pages))",
            r"\b\d{2}\s*(?:yo|y/o|years? old)\b",
            r"[\(\[]\s*\d{2}\s*[MFmf]\s*[\)\]]|[\(\[]\s*[MFmf]\s*\d{2}\s*[\)\]]|\b\d{2}[MF]\b",
            r"\bI was born in (?:19|20)\d{2}\b", r"\bmy (?:\d{2}(?:st|nd|rd|th)) birthday\b",
            r"\bI(?:'m| am) (?:in my )?(?:early |mid |late )?(?:teens|twenties|thirties|forties|fifties|sixties|seventies|20s|30s|40s|50s|60s|70s)\b"],
    'sex_gender': [r"\bI(?:'m| am)\s+(?:a|an)?\s*(?:\w+\s+){0,3}(?:man|woman|guy|girl|dude|lady|male|female|boy|gal|mom|mum|dad|mother|father|wife|husband|grandma|grandmother|grandpa|grandfather|son|daughter|trans\w*|nonbinary|non-binary)\b",
                   r"\bas an?\s+(?:\w+\s+){0,2}(?:man|woman|guy|girl|male|female|mom|mum|dad|mother|father|wife|husband|trans\w*)\b",
                   r"\bmy (?:husband|wife|boyfriend|girlfriend|bf|gf|fianc[eé]e?|partner|ex-?(?:wife|husband|boyfriend|girlfriend))\b"],
    'occupation': [r"\bI(?:'m| am)\s+(?:a|an)\s+(?:\w+\s+){0,3}(?:doctor|physician|md|nurse|rn|surgeon|dentist|pharmacist|lawyer|attorney|paralegal|teacher|professor|engineer|programmer|developer|coder|software|scientist|accountant|cpa|cop|police|officer|firefighter|paramedic|emt|soldier|marine|veteran|vet|chef|cook|driver|trucker|pilot|electrician|plumber|mechanic|contractor|carpenter|farmer|student|journalist|reporter|writer|artist|musician|designer|analyst|consultant|manager|owner|ceo|realtor|banker|trader|therapist|psychologist|social worker|pastor|priest|nanny|cashier|retail|server|bartender|retiree|retired)\b",
                   r"\bI work(?:ed)?\s+(?:as|in|at|for)\b", r"\bmy (?:job|career|boss|employer|coworkers?|clients?|patients?|students?)\b",
                   r"\bin my (?:line of work|profession|field|industry|practice)\b", r"\bI(?:'m| am) retired\b"],
    'location_nationality': [r"\bI live (?:in|on|near|outside)\b", r"\bI(?:'m| am) from\b", r"\bI grew up (?:in|on|near)\b",
                             # a capital letter is required for the place, so these parts are case-sensitive (?-i:...)
                             r"\bI(?:'m| am|m)\s+(?:in|at|near|outside|around|based in|located in)\s+(?:the\s+)?(?-i:[A-Z])[\w.]+",
                             r"\b(?:here in|over here in|down here in|up here in|out here in)\s+(?:the\s+)?(?-i:[A-Z])\w+",
                             r"\banyone (?:in|from|near|around)\s+(?:the\s+)?(?-i:[A-Z])\w+",
                             r"\b(?:I'm|I am|im)\s+(?:in|from)\s+(?-i:[A-Z]{2})\b", r"\bas an?\s+(?:American|Canadian|Brit|British|Englishman|Australian|Aussie|Kiwi|Irish|Scot|Indian|Pakistani|Bangladeshi|Nigerian|German|French|Mexican|Filipino|Russian|Ukrainian|European|Texan|Californian|New Yorker)\b",
                             r"\bI(?:'m| am)\s+(?:an?\s+)?(?:American|Canadian|British|English|Australian|Irish|Scottish|Indian|Pakistani|Bangladeshi|Nigerian|German|French|Mexican|Filipino|Russian|Ukrainian)\b",
                             r"\bmy (?:country|state|province|city|town|county|hometown)\b", r"\bwhere I live\b", r"\bI moved (?:to|from)\b"],
    'voting_politics': [r"\bI (?:voted|vote|will vote|am voting|'m voting|didn't vote|did not vote|caucused)\b",
                        r"\bmy (?:vote|ballot|polling place|precinct|congressman|congresswoman|senator|representative|mp|governor)\b",
                        r"\bI(?:'m| am)\s+(?:a|an)?\s*(?:\w+\s+){0,2}(?:democrat|republican|conservative|liberal|leftist|progressive|socialist|libertarian|independent|tory|labour|centrist|maga|trump supporter)\b",
                        r"\bas an?\s+(?:\w+\s+){0,1}(?:democrat|republican|conservative|liberal|leftist|progressive|socialist|libertarian|independent|centrist|trump supporter)\b",
                        r"\bregistered (?:democrat|republican|independent|voter)\b"],
    'religion_ethnicity': [r"\bI(?:'m| am)\s+(?:a|an)?\s*(?:\w+\s+){0,2}(?:christian|catholic|protestant|mormon|muslim|jewish|jew|hindu|buddhist|sikh|atheist|agnostic|black|white|asian|latino|latina|hispanic|native|indigenous|mixed)\b",
                           r"\bas an?\s+(?:\w+\s+){0,1}(?:christian|catholic|muslim|jew|jewish|hindu|atheist|black|white|asian|latino|latina|hispanic)\s*(?:man|woman|person|guy|girl)?\b"],
    'family': [r"\bmy (?:kids?|son|daughter|children|child|grandkids?|grandchildren|grandson|granddaughter|baby|toddler|teenager)\b",
               r"\bI have (?:a|an|one|two|three|four|five|\d)\s+(?:\w+\s+)?(?:kids?|children|sons?|daughters?|grandkids?)\b",
               r"\bI(?:'m| am)\s+(?:single|married|divorced|widowed|engaged|a widow|a widower|childless|pregnant)\b"],
    'education': [r"\bI (?:have|got|earned|hold|am getting|'m getting)\s+(?:a|an|my)\s+(?:\w+\s+){0,2}(?:degree|phd|doctorate|masters?|bachelors?|mba|jd|md)\b",
                  r"\bI(?:'m| am)\s+(?:in|at)\s+(?:college|university|uni|grad school|med school|law school|high school)\b",
                  r"\bI (?:graduated|dropped out)\b", r"\bwhen I was in (?:college|high school|university)\b"],
    'military': [r"\bI (?:served|was deployed|enlisted)\b", r"\bI(?:'m| am| was) (?:in|a|an) (?:the )?(?:army|navy|air force|marines?|military|veteran|vet)\b",
                 r"\bmy (?:service|unit|deployment|tour)\b"],
}
# One combined case-insensitive regex per category.
COMPILED = {cat: re.compile('|'.join(f'(?:{p})' for p in ps), re.I) for cat, ps in PATTERNS.items()}

# Drop quoted lines (Reddit quotes start with >).
def unquoted(text):
    return '\n'.join(l for l in text.split('\n') if not l.lstrip().startswith('>'))

# The match with up to 220 characters either side, on one line.
def context(text, start, end, pad=220):
    s, e = max(0, start - pad), min(len(text), end + pad)
    return ('…' if s else '') + text[s:e].replace('\n', ' ') + ('…' if e < len(text) else '')

# One row per match: account, item, category, matched words and context. Counts by account and category.
cands = []
for r in items.itertuples():
    t = unquoted(r.text)
    for cat, rx in COMPILED.items():
        for m in rx.finditer(t):
            cands.append(dict(account=r.account, kind=r.kind, id=r.id, created_utc=r.created_utc,
                              subreddit=r.subreddit, category=cat, match=m.group(0),
                              context=context(t, m.start(), m.end()), permalink=r.permalink))
cands = pd.DataFrame(cands)
cands['date'] = pd.to_datetime(cands['created_utc'], unit='s').dt.strftime('%Y-%m-%d')
cands.groupby(['account', 'category']).size().unstack(fill_value=0)

In [ ]:
print('items with any candidate:', cands['id'].nunique(), 'of', len(items))
# Save the items and the candidate matches.
save_new(items, 'case_study_items_v2.parquet')
save_new(cands, 'identity_candidates_v2.parquet')
# plain-text dumps for review, one per account, oldest first
for a, g in cands.sort_values(['account', 'category', 'created_utc']).groupby('account'):
    path = os.path.join(OUT_DIR, f'candidates_{a}_v2.txt')
    if os.path.exists(path):
        print('exists, not overwritten:', path); continue
    with open(path, 'w') as fh:
        for r in g.itertuples():
            fh.write(f'[{r.category}] {r.date} r/{r.subreddit} {r.kind} {r.id}\n    {r.context}\n')
    print('wrote', path)